# Fig. 1G: global 3'UTR-length index in neurons

Per-cell global 3'UTR-length index along the SVZ lineage, with the discrete lengthening
step from neuroblasts to neurons (NB -> neuron Cliff's delta = +0.56, reproducible in both
neuron-containing batches). Mouse data only; the cross-species comparison (organoids) is in
`fig2/development/supp_4_xspecies_wexpr`.

- Kernel: `apa-python`. Effect size only (no p-values).
- Input: `DirMult_neuron_bridge/{neuron_W_global_percell, neuron_W_global_shift}.csv`.
- Output: `.../Fig1/fig1_neuron_global_w/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

PANEL = "fig1_neuron_global_w"
PANEL_W_PX, PANEL_H_PX = 1000, 780
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)
LINE = ["qNSC1", "qNSC2", "aNSC1", "aNSC2", "TAP", "NB", "Neurons"]

OUT = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_neuron_bridge"
)
pc = pd.read_csv(OUT / "neuron_W_global_percell.csv")
shift = pd.read_csv(OUT / "neuron_W_global_shift.csv").set_index("contrast")
CLIFF_NB = float(shift.loc["NB->Neuron", "cliff_delta"])
print("cells:", len(pc), "| Cliff NB->neuron:", round(CLIFF_NB, 3))

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
g = pc.groupby("celltype")["global_W"].agg(["mean", "std", "count"]).reindex(LINE)
g["sem"] = g["std"] / np.sqrt(g["count"])
xp = np.arange(len(LINE))

ax.plot(xp, g["mean"].values, "-", color="#999999", lw=0.9, zorder=1)
for i, ct in enumerate(LINE):
    is_n = ct == "Neurons"
    ax.errorbar(
        xp[i],
        g["mean"].values[i],
        yerr=g["sem"].values[i],
        fmt="D" if is_n else "o",
        ms=7 if is_n else 5,
        lw=1,
        color=S.paper_palette_celltypes[ct],
        capsize=2.5,
        zorder=3,
        markeredgecolor="black" if is_n else "none",
        markeredgewidth=0.6 if is_n else 0,
    )
ax.axvspan(
    xp[-1] - 0.5,
    xp[-1] + 0.5,
    color=S.paper_palette_celltypes["Neurons"],
    alpha=0.10,
    zorder=0,
)
ax.set_xticks(xp)
ax.set_xticklabels(LINE, rotation=45, ha="right")
ax.set_ylabel("3$^\\prime$UTR-length index (per-cell mean)", fontsize=S.FONT_LABEL)
ax.annotate(
    f"neurons lengthen\nCliff's d = {CLIFF_NB:+.2f}",
    xy=(xp[-1], g["mean"].values[-1]),
    xytext=(xp[-1] - 3.7, g["mean"].max()),
    fontsize=S.FONT_LEGEND,
    color=S.paper_palette_celltypes["Neurons"],
    ha="left",
    arrowprops=dict(
        arrowstyle="->", color=S.paper_palette_celltypes["Neurons"], lw=0.8
    ),
)
ax.tick_params(labelsize=S.FONT_TICK)
fig.subplots_adjust(left=0.16, right=0.97, top=0.95, bottom=0.20)
S.save_and_show(fig, PANEL)

## Variant: half-width x quarter-height (grid-uniformized)

The same panel resized onto the shared grid at `HALF_W_IN x QUARTER_H_IN`
(85 x 58.75 mm), so it tiles as one half of a full-width row next to
`Fig1/fig1e_exemplars__hnrnpa2b1_half` (the Hnrnpa2b1-only half-panel, same
size). Changes vs the main panel: size, slightly smaller markers and a 2-line
y-axis label; the main panel above (~85 x 66 mm, off-grid) is unchanged.
Saved as `fig1_neuron_global_w__quarter`.


In [ ]:
# --- Half-width x quarter-height variant (grid-uniformized to match the fig1e half-panel) ---
# Reuses `pc`, `shift`, `CLIFF_NB`, `LINE` from above; only the size, marker sizes and a
# 2-line y-label change so it tiles next to fig1e_exemplars__hnrnpa2b1_half (HALF_W x QUARTER_H).
PANEL_Q = "fig1_neuron_global_w__quarter"

fig, ax = plt.subplots(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))  # 85 x 58.75 mm
g = pc.groupby("celltype")["global_W"].agg(["mean", "std", "count"]).reindex(LINE)
g["sem"] = g["std"] / np.sqrt(g["count"])
xp = np.arange(len(LINE))

ax.plot(xp, g["mean"].values, "-", color="#999999", lw=0.9, zorder=1)
for i, ct in enumerate(LINE):
    is_n = ct == "Neurons"
    ax.errorbar(
        xp[i],
        g["mean"].values[i],
        yerr=g["sem"].values[i],
        fmt="D" if is_n else "o",
        ms=6 if is_n else 4.5,
        lw=1,
        color=S.paper_palette_celltypes[ct],
        capsize=2.5,
        zorder=3,
        markeredgecolor="black" if is_n else "none",
        markeredgewidth=0.6 if is_n else 0,
    )
ax.axvspan(
    xp[-1] - 0.5,
    xp[-1] + 0.5,
    color=S.paper_palette_celltypes["Neurons"],
    alpha=0.10,
    zorder=0,
)
ax.set_xticks(xp)
ax.set_xticklabels(LINE, rotation=45, ha="right")
ax.set_ylabel("3$^\\prime$UTR-length index\n(per-cell mean)", fontsize=S.FONT_LABEL)
ax.annotate(
    f"neurons lengthen\nCliff's d = {CLIFF_NB:+.2f}",
    xy=(xp[-1], g["mean"].values[-1]),
    xytext=(xp[-1] - 3.7, g["mean"].max()),
    fontsize=S.FONT_LEGEND,
    color=S.paper_palette_celltypes["Neurons"],
    ha="left",
    arrowprops=dict(
        arrowstyle="->", color=S.paper_palette_celltypes["Neurons"], lw=0.8
    ),
)
ax.tick_params(labelsize=S.FONT_TICK)
fig.subplots_adjust(left=0.20, right=0.97, top=0.96, bottom=0.24)
S.save_and_show(fig, PANEL_Q)

### Variant: quarter-width x quarter-height (no annotation; tiles next to fig1f quarter)


In [ ]:
# --- Quarter-width x quarter-height variant (to tile next to the quarter-width fig1f) ---
# Same as the __quarter panel but quarter width (40.5 mm) and without the "neurons lengthen"
# annotation. Reuses pc / LINE.
PANEL_QQ = "fig1_neuron_global_w__quarter_quarter"

fig, ax = plt.subplots(figsize=(S.QUARTER_W_IN, S.QUARTER_H_IN))  # 40.5 x 58.75 mm
g = pc.groupby("celltype")["global_W"].agg(["mean", "std", "count"]).reindex(LINE)
g["sem"] = g["std"] / np.sqrt(g["count"])
xp = np.arange(len(LINE))
ax.plot(xp, g["mean"].values, "-", color="#999999", lw=0.8, zorder=1)
for i, ct in enumerate(LINE):
    is_n = ct == "Neurons"
    ax.errorbar(
        xp[i],
        g["mean"].values[i],
        yerr=g["sem"].values[i],
        fmt="D" if is_n else "o",
        ms=5 if is_n else 3.5,
        lw=0.9,
        color=S.paper_palette_celltypes[ct],
        capsize=2,
        markeredgecolor="black" if is_n else "none",
        markeredgewidth=0.5 if is_n else 0,
        zorder=3,
    )
ax.axvspan(
    xp[-1] - 0.5,
    xp[-1] + 0.5,
    color=S.paper_palette_celltypes["Neurons"],
    alpha=0.10,
    zorder=0,
)
ax.set_xticks(xp)
ax.set_xticklabels(LINE, rotation=45, ha="right", fontsize=S.FONT_TICK - 1)
ax.set_ylabel(
    "3$^\\prime$UTR-length index\n(per-cell mean)", fontsize=S.FONT_LABEL - 0.5
)
ax.tick_params(labelsize=S.FONT_TICK - 1)
fig.subplots_adjust(left=0.30, right=0.95, top=0.96, bottom=0.30)
S.save_and_show(fig, PANEL_QQ)